# Lab 10: Spring Boot with Iceberg - Solution

This notebook provides a Python-based solution demonstrating the concepts from Lab 10 (Spring Boot with Iceberg). While the actual lab uses Java/Spring Boot, this solution shows equivalent patterns using Python and PyIceberg.

## Setup

Configure the environment for Iceberg operations.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
import json

# Create Spark session with Iceberg support
spark = SparkSession.builder \
    .appName("Spring-Boot-Iceberg-Solution") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://polaris:8181/api/catalog") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

## Part 1: Domain Models (Python Classes)

In Spring Boot, you would use Java classes with JPA annotations. Here we use Python classes and PySpark schemas.

In [ ]:
# Define schemas equivalent to Java domain models
customer_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("address", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

product_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("description", StringType(), True),
    StructField("price", DecimalType(10, 2), False),
    StructField("category", StringType(), True),
    StructField("stock_quantity", IntegerType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

order_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("total_amount", DecimalType(10, 2), False),
    StructField("shipping_address", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

print("Domain schemas defined successfully")

## Part 2: Repository Layer (CRUD Operations)

In Spring Boot, you would use Spring Data JPA repositories. Here we implement CRUD operations using Spark.

In [ ]:
class IcebergRepository:
    """Base repository class for Iceberg operations"""
    
    def __init__(self, table_name, schema):
        self.table_name = table_name
        self.schema = schema
        self.full_table_name = f"iceberg.demo.{table_name}"
    
    def initialize_table(self):
        """Create the table if it doesn't exist"""
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.full_table_name} (
                id INT,
                name STRING,
                email STRING,
                phone STRING,
                address STRING,
                created_at TIMESTAMP,
                updated_at TIMESTAMP
            )
            USING iceberg
        """)
        print(f"Table {self.table_name} initialized")
    
    def find_all(self):
        """Find all records"""
        return spark.table(self.full_table_name)
    
    def find_by_id(self, id):
        """Find a record by ID"""
        return spark.table(self.full_table_name).filter(col("id") == id)
    
    def save(self, data):
        """Save a single record"""
        df = spark.createDataFrame([data], schema=self.schema)
        df.write.format("iceberg").mode("append").saveAsTable(self.full_table_name)
        print(f"Record saved to {self.table_name}")
    
    def save_all(self, data_list):
        """Save multiple records"""
        df = spark.createDataFrame(data_list, schema=self.schema)
        df.write.format("iceberg").mode("append").saveAsTable(self.full_table_name)
        print(f"{len(data_list)} records saved to {self.table_name}")
    
    def update(self, id, data):
        """Update a record by ID"""
        # In Iceberg, we use MERGE for updates
        temp_view = "temp_update_view"
        df = spark.createDataFrame([data], schema=self.schema)
        df.createOrReplaceTempView(temp_view)
        
        spark.sql(f"""
            MERGE INTO {self.full_table_name} AS target
            USING {temp_view} AS source
            ON target.id = source.id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"Record {id} updated in {self.table_name}")
    
    def delete(self, id):
        """Delete a record by ID"""
        spark.sql(f"DELETE FROM {self.full_table_name} WHERE id = {id}")
        print(f"Record {id} deleted from {self.table_name}")

print("Repository class defined successfully")

## Part 3: Service Layer (Business Logic)

In Spring Boot, you would use @Service classes. Here we implement service logic.

In [ ]:
class CustomerService:
    """Service class for customer operations"""
    
    def __init__(self):
        self.repository = IcebergRepository("customers", customer_schema)
    
    def initialize_table(self):
        self.repository.initialize_table()
    
    def find_all(self):
        return self.repository.find_all()
    
    def find_by_id(self, id):
        return self.repository.find_by_id(id)
    
    def create(self, customer_data):
        """Create a new customer with timestamps"""
        customer_data['created_at'] = datetime.now()
        customer_data['updated_at'] = datetime.now()
        self.repository.save(customer_data)
        return customer_data
    
    def update(self, id, customer_data):
        """Update an existing customer"""
        customer_data['updated_at'] = datetime.now()
        self.repository.update(id, customer_data)
        return customer_data
    
    def delete(self, id):
        self.repository.delete(id)

class ProductService:
    """Service class for product operations"""
    
    def __init__(self):
        self.repository = IcebergRepository("products", product_schema)
    
    def initialize_table(self):
        self.repository.initialize_table()
    
    def find_all(self):
        return self.repository.find_all()
    
    def find_by_id(self, id):
        return self.repository.find_by_id(id)
    
    def create(self, product_data):
        product_data['created_at'] = datetime.now()
        product_data['updated_at'] = datetime.now()
        self.repository.save(product_data)
        return product_data
    
    def update(self, id, product_data):
        product_data['updated_at'] = datetime.now()
        self.repository.update(id, product_data)
        return product_data
    
    def delete(self, id):
        self.repository.delete(id)

print("Service classes defined successfully")

## Part 4: REST API Layer (Simulated)

In Spring Boot, you would use @RestController classes. Here we simulate API operations.

In [ ]:
class CustomerController:
    """Controller class simulating REST API endpoints"""
    
    def __init__(self):
        self.service = CustomerService()
    
    def initialize_table(self):
        """POST /api/customers/initialize"""
        return self.service.initialize_table()
    
    def get_all(self):
        """GET /api/customers"""
        return self.service.find_all()
    
    def get_by_id(self, id):
        """GET /api/customers/{id}"""
        result = self.service.find_by_id(id)
        if result.count() == 0:
            return None  # 404 Not Found
        return result
    
    def create(self, customer_data):
        """POST /api/customers"""
        # Validate data (equivalent to @Valid annotation)
        if not customer_data.get('name') or not customer_data.get('email'):
            raise ValueError("Name and email are required")
        return self.service.create(customer_data)
    
    def update(self, id, customer_data):
        """PUT /api/customers/{id}"""
        if self.service.find_by_id(id).count() == 0:
            return None  # 404 Not Found
        customer_data['id'] = id
        return self.service.update(id, customer_data)
    
    def delete(self, id):
        """DELETE /api/customers/{id}"""
        self.service.delete(id)
        return True  # 204 No Content

class ProductController:
    """Controller class for product operations"""
    
    def __init__(self):
        self.service = ProductService()
    
    def initialize_table(self):
        """POST /api/products/initialize"""
        return self.service.initialize_table()
    
    def get_all(self):
        """GET /api/products"""
        return self.service.find_all()
    
    def get_by_id(self, id):
        """GET /api/products/{id}"""
        result = self.service.find_by_id(id)
        if result.count() == 0:
            return None
        return result
    
    def create(self, product_data):
        """POST /api/products"""
        if not product_data.get('name') or not product_data.get('price'):
            raise ValueError("Name and price are required")
        return self.service.create(product_data)
    
    def update(self, id, product_data):
        """PUT /api/products/{id}"""
        if self.service.find_by_id(id).count() == 0:
            return None
        product_data['id'] = id
        return self.service.update(id, product_data)
    
    def delete(self, id):
        """DELETE /api/products/{id}"""
        self.service.delete(id)
        return True

print("Controller classes defined successfully")

## Part 5: Testing the Application

In [ ]:
# Initialize controllers
customer_controller = CustomerController()
product_controller = ProductController()

# Initialize tables
print("Initializing tables...")
customer_controller.initialize_table()
product_controller.initialize_table()

In [ ]:
# Create customers (POST /api/customers)
print("Creating customers...")
customer1 = {
    'id': 1,
    'name': 'John Doe',
    'email': 'john@example.com',
    'phone': '+1-555-0101',
    'address': '123 Main St, NYC, NY 10001'
}
customer_controller.create(customer1)

customer2 = {
    'id': 2,
    'name': 'Jane Smith',
    'email': 'jane@example.com',
    'phone': '+1-555-0102',
    'address': '456 Oak Ave, LA, CA 90001'
}
customer_controller.create(customer2)

In [ ]:
# Create products (POST /api/products)
print("Creating products...")
product1 = {
    'id': 1,
    'name': 'Laptop Pro 15"',
    'description': 'High-performance laptop with 16GB RAM and 512GB SSD',
    'price': 1299.99,
    'category': 'Electronics',
    'stock_quantity': 50
}
product_controller.create(product1)

product2 = {
    'id': 2,
    'name': 'Wireless Mouse',
    'description': 'Ergonomic wireless mouse with precision tracking',
    'price': 29.99,
    'category': 'Electronics',
    'stock_quantity': 200
}
product_controller.create(product2)

In [ ]:
# Get all customers (GET /api/customers)
print("All customers:")
customer_controller.get_all().show()

In [ ]:
# Get customer by ID (GET /api/customers/1)
print("Customer with ID 1:")
customer_controller.get_by_id(1).show()

In [ ]:
# Update customer (PUT /api/customers/1)
print("Updating customer 1...")
updated_customer = {
    'name': 'John Smith',
    'email': 'john.smith@example.com',
    'phone': '+1-555-0103',
    'address': '789 Pine Rd, Chicago, IL 60601'
}
customer_controller.update(1, updated_customer)

In [ ]:
# Verify update
print("Updated customer:")
customer_controller.get_by_id(1).show()

In [ ]:
# Delete customer (DELETE /api/customers/2)
print("Deleting customer 2...")
customer_controller.delete(2)

In [ ]:
# Verify deletion
print("Remaining customers:")
customer_controller.get_all().show()

## Part 6: Data Validation

In [ ]:
# Test validation (equivalent to @Valid annotation)
try:
    invalid_customer = {
        'id': 3,
        'name': '',  # Empty name should fail validation
        'email': 'invalid-email'  # Invalid email should fail validation
    }
    customer_controller.create(invalid_customer)
except ValueError as e:
    print(f"Validation error: {e}")

## Part 7: Error Handling

In [ ]:
# Test error handling (equivalent to @ExceptionHandler)
print("Testing error handling...")

# Try to get non-existent customer
result = customer_controller.get_by_id(999)
if result is None:
    print("404 Not Found: Customer does not exist")

# Try to update non-existent customer
result = customer_controller.update(999, {'name': 'Test'})
if result is None:
    print("404 Not Found: Customer does not exist")

## Part 8: Performance Optimization (Caching)

In [ ]:
# Simulate caching (equivalent to @Cacheable)
class CachedCustomerService(CustomerService):
    """Service with caching simulation"""
    
    def __init__(self):
        super().__init__()
        self.cache = {}  # Simple in-memory cache
    
    def find_by_id(self, id):
        """Find customer with caching"""
        if id in self.cache:
            print(f"Cache hit for customer {id}")
            return self.cache[id]
        
        print(f"Cache miss for customer {id}")
        result = super().find_by_id(id)
        self.cache[id] = result
        return result
    
    def update(self, id, customer_data):
        """Update customer and invalidate cache"""
        result = super().update(id, customer_data)
        if id in self.cache:
            del self.cache[id]  # Invalidate cache
        return result

# Test caching
cached_service = CachedCustomerService()
print("Testing caching...")
cached_service.find_by_id(1)  # Cache miss
cached_service.find_by_id(1)  # Cache hit

## Part 9: Transaction Management

In [ ]:
# Simulate transaction management (equivalent to @Transactional)
def create_order_with_transaction(customer_id, product_data, order_data):
    """Create order with transaction-like behavior"""
    try:
        # In a real Spring Boot app, this would be within a @Transactional method
        print("Starting transaction...")
        
        # Step 1: Validate customer exists
        customer = customer_controller.get_by_id(customer_id)
        if customer is None:
            raise ValueError("Customer not found")
        
        # Step 2: Create product
        product_controller.create(product_data)
        
        # Step 3: Create order (simulated)
        print(f"Order created for customer {customer_id}")
        
        print("Transaction committed successfully")
        return True
    
    except Exception as e:
        print(f"Transaction rolled back: {e}")
        # In a real app, you would rollback changes here
        return False

# Test transaction
print("Testing transaction management...")
new_product = {
    'id': 3,
    'name': 'Test Product',
    'description': 'Test description',
    'price': 99.99,
    'category': 'Test',
    'stock_quantity': 10
}

create_order_with_transaction(1, new_product, {})

## Cleanup

In [ ]:
# Drop tables
spark.sql("DROP TABLE IF EXISTS iceberg.demo.customers")
spark.sql("DROP TABLE IF EXISTS iceberg.demo.products")

print("Tables dropped successfully")

## Summary

This solution demonstrated Spring Boot patterns using Python:

1. **Domain Models**: Defined schemas equivalent to Java entity classes
2. **Repository Layer**: Implemented CRUD operations similar to Spring Data JPA
3. **Service Layer**: Added business logic and validation
4. **Controller Layer**: Simulated REST API endpoints
5. **Data Validation**: Implemented validation logic
6. **Error Handling**: Added proper error handling patterns
7. **Caching**: Demonstrated caching for performance optimization
8. **Transaction Management**: Simulated transactional operations

In a production Spring Boot application, you would:
- Use actual Spring Boot framework with @RestController, @Service, @Repository
- Implement proper REST endpoints with Spring MVC
- Use Spring Data JPA for database operations
- Add Spring Security for authentication and authorization
- Implement proper transaction management with @Transactional
- Add caching with Spring Cache abstraction
- Use Spring Boot Actuator for monitoring
- Add proper exception handling with @ControllerAdvice
- Implement API documentation with Swagger/OpenAPI